In [2]:
import torch
import torch.nn as nn

device = torch.device("mps")
print(f"using device: {device}")

using device: mps


In [3]:
# Toy dimensions

L = 64 # sequence length
N_seq = 8 # number of seq in MSA
c_m = 64 # MSA representation dims
c_z = 32 # pair representation dims

print (f"MSA repr shape: ({N_seq}, {L}, {c_m})")
print (f"Pair repr shape: ({L}, {L}, {c_z})")

MSA repr shape: (8, 64, 64)
Pair repr shape: (64, 64, 32)


In [4]:
aatype = torch.randint(0, 21, (L,), device=device)
residue_index = torch.arange(L, device=device)
msa = torch.randint(0, 23, (N_seq, L), device=device)

print("aatype:        ", aatype.shape)       # (L,)
print("residue_index: ", residue_index.shape) # (L,)
print("msa:           ", msa.shape)           # (N_seq, L)


aatype:         torch.Size([64])
residue_index:  torch.Size([64])
msa:            torch.Size([8, 64])


In [5]:
print(residue_index)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63], device='mps:0')


In [6]:
import torch.nn.functional as F

msa_one_hot = F.one_hot(msa, num_classes=23).float() # (N_seq, L, 23)
print("msa_one_hot:", msa_one_hot.shape)


msa_one_hot: torch.Size([8, 64, 23])


In [7]:
msa

tensor([[ 0, 16, 14, 21,  0,  4, 15, 11, 18, 11,  6,  5, 13, 18,  1, 22, 10,  8,
          6, 20,  8, 19,  8,  3,  3,  2,  1,  9,  9, 20,  1, 10,  5,  2,  5,  6,
         21,  2,  8,  6,  7, 11,  7, 12, 10,  7,  2, 10, 13,  5,  4,  1,  8,  0,
          3, 14,  4, 11,  5, 16,  4, 12, 20, 14],
        [ 2,  1, 18, 20,  9,  4, 11, 21,  9,  1,  6, 10, 18, 13,  7, 15,  0, 15,
         14, 11, 22,  1, 19, 21, 14,  2, 21,  2, 13, 18,  9, 12, 12, 12,  7,  0,
          1,  1, 10,  2, 11, 18,  1,  4, 10, 21, 13, 15,  3,  0, 16, 16, 21,  1,
         16,  4,  5, 19, 16,  4, 16, 10, 13, 21],
        [12,  7, 14, 14,  1, 13, 22,  8, 12, 12, 16, 19,  9, 15, 11, 19,  6, 11,
         12,  4,  5,  0, 22, 16,  7, 22, 20,  3, 13, 12, 14,  0,  5, 17, 22, 10,
         17, 17,  5,  7,  0,  4,  4,  8,  5, 13,  0,  9, 17, 12, 13,  1, 14, 17,
          1,  1,  4, 18,  4,  8,  0,  6,  3, 16],
        [10, 16,  6, 20, 12,  2, 17, 19,  2, 19,  7, 12, 19,  2,  2,  4,  6,  3,
         15, 16, 13,  5,  9,  8, 14, 12,

In [8]:
class InputEmbedder(nn.Module):
    def __init__(self, c_m, c_z):
        super().__init__()
        self.msa_embedder = nn.Linear(23, c_m)
        self.pair_embedder = nn.Linear(65, c_z)

    def forward(self, msa, residue_index):
        # msa: (N_seq, L), residue_index: (L,)

        #MSA
        msa_one_hot = F.one_hot(msa, num_classes=23).float()
        m = self.msa_embedder(msa_one_hot)

        #Pair
        rel_pos = residue_index[None, :] - residue_index[:,None] #(L, L)
        rel_pos_one_hot = F.one_hot(rel_pos.clamp(-32, 32) + 32, num_classes=65).float()
        z = self.pair_embedder(rel_pos_one_hot)

        return m, z
    

embedder = InputEmbedder(c_m, c_z).to(device)
m, z = embedder(msa, residue_index)
print("m:", m.shape)
print("z:", z.shape)

m: torch.Size([8, 64, 64])
z: torch.Size([64, 64, 32])


In [9]:
c_head = 8 # dimensions per attention head
num_heads = 4

to_q = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)
to_k = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)
to_v = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)

pair_bias = nn.Linear(c_z, num_heads, bias=False).to(device)

row = m[0]
q = to_q(row)
k = to_k(row)
v = to_v(row)

print("q:", q.shape)
print("Pair bias input z:", z.shape)
print("Pair bias output:", pair_bias(z).shape)

q: torch.Size([64, 32])
Pair bias input z: torch.Size([64, 64, 32])
Pair bias output: torch.Size([64, 64, 4])


In [10]:
pair_bias

Linear(in_features=32, out_features=4, bias=False)

In [11]:
from einops import rearrange

q = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)
k = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)
v = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)

print('q:', q.shape)

# Raw attention scores: for each head, dot every query with every key
scale = c_head ** - 0.5
scores = torch.einsum('h i d, h j d -> h i j', q, k) * scale
print("scores:", scores.shape)

#pair bias
bias = rearrange(pair_bias(z), 'i j h -> h i j')
print("bias:", bias.shape)

scores = scores + bias

q: torch.Size([4, 64, 8])
scores: torch.Size([4, 64, 64])
bias: torch.Size([4, 64, 64])


In [12]:
# softmax over the j axis, for each query pos i, weights over all key positions, j to sum to 1
attn_weights = torch.softmax(scores, dim=-1)
print("attn_weights:", attn_weights.shape)
print("row sums (should all be 1.0):", attn_weights[0, 0].sum().item())


# aggregate values: for each position i, weighted sum of all value vectors
out = torch.einsum('h i j, h j d -> h i d', attn_weights, v)
print("out:", out.shape) # (L, num_heads*c_head)

out = rearrange(out, 'h l d -> l (h d)')
print("out merged:", out.shape)

# final linear projection back to c_m
to_out = nn.Linear(num_heads * c_head, c_m).to(device)
out = to_out(out)
print("out projected:", out.shape)

attn_weights: torch.Size([4, 64, 64])
row sums (should all be 1.0): 1.0
out: torch.Size([4, 64, 8])
out merged: torch.Size([64, 32])
out projected: torch.Size([64, 64])


In [13]:
to_gate = nn.Linear(c_m, num_heads * c_head).to(device)

gate = torch.sigmoid(to_gate(row))
gate = rearrange(gate, 'l (h d) -> h l d', h = num_heads)

# recompute out before merging
out_heads = torch.einsum('h i j, h j d -> h i d', attn_weights, v)

# apply gate element wise
out_gated = gate * out_heads

# merge and project
out_gated = rearrange(out_gated, 'h l d -> l (h d)')
out_gated = to_out(out_gated)
print('gated output:', out_gated.shape)

gated output: torch.Size([64, 64])


In [14]:
class MSARowAttention(nn.Module):
    def __init__(self, c_m, c_z, c_head, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.c_head = c_head
        self.norm = nn.LayerNorm(c_m)
        self.to_q = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_k = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_v = nn.Linear(c_m, c_head * num_heads, bias=False)

        self.to_gate = nn.Linear(c_m, c_head * num_heads)
        self.pair_bias = nn.Linear(c_z, num_heads, bias=False)
        self.to_out = nn.Linear(c_head * num_heads, c_m)

    def forward(self, m, z):
        # m: (N_seq, L, c_m), z: (L, L, c_z)
        m = self.norm(m)
        scale = self.c_head ** -0.5
        bias = rearrange(self.pair_bias(z), 'i j h -> h i j') # (H, L, L)

        q = rearrange(self.to_q(m), 'n l (h d) -> h n l d', h = self.num_heads)
        k = rearrange(self.to_k(m), 'n l (h d) -> h n l d', h = self.num_heads)
        v = rearrange(self.to_v(m), 'n l (h d) -> h n l d', h = self.num_heads)

        scores = torch.einsum('h n i d, h n j d -> h n i j', q, k) * scale
        scores = scores + bias.unsqueeze(1) # broadcast over N_seq
        attn = torch.softmax(scores, dim=-1) # (H, N_seq, L, L)

        out = torch.einsum('h n i j, h n j d -> h n i d', attn, v)
        gate = torch.sigmoid(rearrange(self.to_gate(m), 'n l (h d) -> h n l d', h = self.num_heads))
        out = rearrange(gate * out, 'h n l d -> n l (h d)')
        return self.to_out(out) # (N_seq, L, c_m)

row_attn = MSARowAttention(c_m, c_z, c_head=8, num_heads=4).to(device)
out = row_attn(m,z)
print('output:', out.shape)

output: torch.Size([8, 64, 64])


In [15]:
m_new = m + row_attn(m,z)
print("m_new:", m_new.shape)

m_new: torch.Size([8, 64, 64])


In [16]:

class MSAColumnAttention(nn.Module):
    def __init__(self, c_m, c_head, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.c_head = c_head
        self.norm = nn.LayerNorm(c_m)
        self.to_q = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_k = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_v = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_gate = nn.Linear(c_m, c_head * num_heads)
        self.to_out = nn.Linear(c_head * num_heads, c_m)

    def forward(self, m):
        # m: (N_seq, L, c_m)
        m = self.norm(m)
        scale = self.c_head **-0.5

        # transpose so L is the outer dim and attend across N_seq for each position
        m_t = rearrange(m, 'n l d -> l n d') # (L, N_seq, c_m)

        q = rearrange(self.to_q(m_t), 'l n (h d) -> h l n d', h = self.num_heads)
        k = rearrange(self.to_k(m_t), 'l n (h d) -> h l n d', h = self.num_heads)
        v = rearrange(self.to_v(m_t), 'l n (h d) -> h l n d', h = self.num_heads)

        scores = torch.einsum('h l i d, h l j d -> h l i j', q, k) * scale
        attn = torch.softmax(scores, dim=-1) # H, L, N_seq, N_seq

        out = torch.einsum('h l i j, h l j d -> h l i d', attn, v)
        gate = torch.sigmoid(rearrange(self.to_gate(m_t), 'l n (h d) -> h l n d', h=self.num_heads))
        out = rearrange(gate * out, 'h l n d -> n l (h d)')
        return self.to_out(out)

col_attn = MSAColumnAttention(c_m, c_head=8, num_heads=4).to(device)
m_new = m + col_attn(m)
print('m_new:', m_new.shape) # 8, 64, 64

m_new: torch.Size([8, 64, 64])


In [17]:
class MSATransition(nn.Module):
    def __init__(self, c_m, expansion=4):
        super().__init__()
        self.norm = nn.LayerNorm(c_m)
        self.ff = nn.Sequential(
            nn.Linear(c_m, c_m * expansion),
            nn.ReLU(),
            nn.Linear(c_m * expansion, c_m)
        )

    def forward(self, m):
        return self.ff(self.norm(m)) # N_seq, L, c_m

transition = MSATransition(c_m).to(device)
m_new = m + transition(m)
print("m_new:", m_new.shape)   # (8, 64, 64)

m_new: torch.Size([8, 64, 64])


In [18]:
c_hidden_opm = 16 # project down before outer product

linear_a = nn.Linear(c_m, c_hidden_opm).to(device)
linear_b = nn.Linear(c_m, c_hidden_opm).to(device)

a = linear_a(m) # N_seq, L, c_hidden_opm
b = linear_b(m)

print("a:", a.shape)

a: torch.Size([8, 64, 16])


In [19]:
# outer product: for every pair(i, j), combine a[:,i,:] with b[:, j, :]
# einsum for each sequence s, position i, position j, outer product of a[s, i, and b[s, j]

outer = torch.einsum('s i d, s j e -> s i j d e', a, b) # n_seq, L, L, c_hidden, c_hidden
print("outer:", outer.shape)

# mean over sequences, covarying patterns survive
outer_mean = outer.mean(dim=0)
print("outer_mean:", outer_mean.shape)



outer: torch.Size([8, 64, 64, 16, 16])
outer_mean: torch.Size([64, 64, 16, 16])


In [22]:
class OuterProductMean(nn.Module):
    def __init__(self, c_m, c_z, c_hidden=16):
        super().__init__()
        self.norm = nn.LayerNorm(c_m)
        self.linear_a = nn.Linear(c_m, c_hidden)
        self.linear_b = nn.Linear(c_m, c_hidden)
        self.linear_out = nn.Linear(c_hidden*c_hidden, c_z)

    def forward(self, m, z):
        m = self.norm(m)
        a = self.linear_a(m)
        b = self.linear_b(m)
        outer = torch.einsum('s i d, s j e -> s i j d e', a, b).mean(dim=0)
        z = z + self.linear_out(rearrange(outer, 'i j d e -> i j (d e)'))
        return z

opm = OuterProductMean(c_m, c_z).to(device)
z_new = opm(m, z)
print("z_new:", z_new.shape)   # (64, 64, 32)

z_new: torch.Size([64, 64, 32])


In [ ]:
z

tensor([[[ 0.1173, -0.0333,  0.0285,  ...,  0.0125, -0.0248, -0.1050],
         [ 0.0217,  0.0212,  0.1461,  ..., -0.1319,  0.1406, -0.0389],
         [ 0.1598,  0.0538,  0.1174,  ...,  0.0161, -0.0249, -0.1159],
         ...,
         [ 0.0277, -0.0339,  0.1320,  ..., -0.0343,  0.0739, -0.1716],
         [ 0.0277, -0.0339,  0.1320,  ..., -0.0343,  0.0739, -0.1716],
         [ 0.0277, -0.0339,  0.1320,  ..., -0.0343,  0.0739, -0.1716]],

        [[ 0.0781,  0.0294, -0.0222,  ..., -0.1671,  0.1497, -0.1247],
         [ 0.1173, -0.0333,  0.0285,  ...,  0.0125, -0.0248, -0.1050],
         [ 0.0217,  0.0212,  0.1461,  ..., -0.1319,  0.1406, -0.0389],
         ...,
         [ 0.0277, -0.0339,  0.1320,  ..., -0.0343,  0.0739, -0.1716],
         [ 0.0277, -0.0339,  0.1320,  ..., -0.0343,  0.0739, -0.1716],
         [ 0.0277, -0.0339,  0.1320,  ..., -0.0343,  0.0739, -0.1716]],

        [[-0.0607, -0.1244, -0.0340,  ..., -0.1220, -0.0305, -0.0373],
         [ 0.0781,  0.0294, -0.0222,  ..., -0

In [23]:
c_hidden_tri = 32

linear_a = nn.Linear(c_z, c_hidden_tri).to(device)
linear_b = nn.Linear(c_z, c_hidden_tri).to(device)
linear_g = nn.Linear(c_z, c_hidden_tri).to(device) # gate

a = linear_a(z_new)
b = linear_b(z_new)
g = torch.sigmoid(linear_g(z_new))

print("a:", a.shape)
print("g:", g.shape)

a: torch.Size([64, 64, 32])
g: torch.Size([64, 64, 32])
